In [ ]:
import dotenv
import tqdm
from motor.motor_asyncfo irom moAsyncIOMotorClientmotor_asyncfo irom moAsyncIOMotorClientmotor_asyncio import AsyncIOMotorClientmotor_asyncio import AsyncIOMotorClient

In [7]:
env = dotenv.dotenv_values()

In [8]:
client = AsyncIOMotorClient(env["OLD_MONGO_DSN"])
old_db = client["ontologies"]
collection = old_db["anatomic_locations"]
await collection.count_documents({})

2901

In [9]:
# Make a client to NEW_MONGO_DSN and ping to make sure we can connect
client = AsyncIOMotorClient(env["NEW_MONGO_DSN"])
new_db = client["ontologies"]

In [16]:
await new_db["snomedct"].count_documents({})

205600

In [ ]:
# For each collection, copy the documents from the old collection to the new collection in batches of 100, but
# drop the "embedding_vector" column from each document
async def migrate_collection(collection_name: str, /, batch_size: int = 500, initial_skip: int = 0):
    old_collection = old_db[collection_name]
    new_collection = new_db[collection_name]
    count = await old_collection.count_documents({})

    async def get_batch():
        skip = initial_skip
        while skip < count:
            batch = await old_collection.find({}).skip(skip).limit(batch_size).to_list(length=batch_size)
            for document in batch:
                document.pop("embedding_vector", None)
            skip += len(batch)
            yield batch

    with tqdm.tqdm(total=count, initial=initial_skip) as pbar:
        async for batch in get_batch():
            await new_collection.insert_many(batch)
            pbar.update(len(batch))

In [18]:
await migrate_collection("snomedct", initial_skip=206600, batch_size=5000)

100%|██████████| 508540/508540 [47:01<00:00, 107.03it/s]
